# 🚀 Master Training Notebook: Huấn Luyện Trọn Gói (Agent 1 + Agent 3 + Agent 4)
**Tự động 100%: Clone GitHub $\rightarrow$ Huấn luyện hết 4 bộ Trọng số $\rightarrow$ Tự động lưu thẳng về Link Google Drive & GitHub của bạn (Không cần tải thủ công)**

### Hướng dẫn chỉ với 2 bước:
1. Tại **Ô CẤU HÌNH (Bước 1)** bên dưới, dán **Link Thư mục Google Drive** của bạn vào biến `GOOGLE_DRIVE_FOLDER_LINK` (hoặc điền `GITHUB_TOKEN` nếu muốn đẩy thẳng trọng số mới về GitHub).
2. Bấm **Run All (Chạy tất cả)**: Hệ thống sẽ tự động giải phóng VRAM giữa từng chặng để chạy mượt cả 4 mô hình (`router`, `generator` SFT+DPO, `critic`, `fact_checker_nli`) trong cùng 1 phiên và tự động đẩy kết quả về Drive/GitHub.


In [ ]:
# ==============================================================================
# BƯỚC 1: Ô CẤU HÌNH DUY NHẤT (DÁN LINK GOOGLE DRIVE CỦA BẠN VÀO ĐÂY)
# ==============================================================================

# 1. Dán Link Thư mục Google Drive của bạn (Ví dụ: "https://drive.google.com/drive/folders/1AbCdEfGhIjKlMnOpQrStUvWxYz")
GOOGLE_DRIVE_FOLDER_LINK = ""

# 2. (Tùy chọn) Tên thư mục lưu trên Google Drive nếu bạn chạy bằng Google Colab:
COLAB_DRIVE_SAVE_PATH = "/content/drive/MyDrive/MultiAgent_RAG_Weights"

# 3. (Tùy chọn - Rất khuyên dùng trên Kaggle) Điền GitHub Personal Access Token (ghp_...)
#    để sau khi train xong, Notebook TỰ ĐỘNG git push trọng số FP16 (<80MB) về thẳng GitHub của bạn!
GITHUB_TOKEN = ""

# 4. Chọn Agent muốn huấn luyện: "all" (chạy hết cả 4 bộ trọng số), hoặc "nli", "generator", "critic", "router"
TRAIN_TARGET = "all"
EPOCHS = 2


In [ ]:
# BƯỚC 2: CÀI ĐẶT THƯ VIỆN & TỰ ĐỘNG CLONE DỰ ÁN TỪ GITHUB
!pip install -q -U torch transformers peft trl bitsandbytes datasets accelerate sentencepiece gdown PyDrive2 huggingface_hub pymupdf safetensors

import os
import re
import subprocess

REPO_URL = "https://github.com/ducmanh-jr/Multi-Agent_RAG.git"
PROJECT_DIR = "/content/Multi-Agent_RAG" if os.path.exists("/content") else "/kaggle/working/Multi-Agent_RAG"

if not os.path.exists(PROJECT_DIR):
    print(f"[GitHub] Đang clone dự án từ {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    print(f"[GitHub] Đang cập nhật (git pull) tại {PROJECT_DIR} ...")
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=False)

os.chdir(PROJECT_DIR)

def extract_drive_id(url_or_id: str) -> str:
    if not url_or_id:
        return ""
    m = re.search(r"(?:folders/|/d/|id=)([a-zA-Z0-9_-]{15,})", url_or_id)
    return m.group(1) if m else url_or_id.strip()

DRIVE_FOLDER_ID = extract_drive_id(GOOGLE_DRIVE_FOLDER_LINK)
print("[OK] Thư mục dự án:", os.getcwd())
if DRIVE_FOLDER_ID:
    print("[OK] Đã nhận diện Google Drive Folder ID:", DRIVE_FOLDER_ID)


In [ ]:
# BƯỚC 3: KẾT NỐI GOOGLE DRIVE TRƯỚC KHI TRAIN
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(COLAB_DRIVE_SAVE_PATH, exist_ok=True)
    IN_COLAB = True
    print(f"[Google Colab] Đã kết nối Google Drive tại: {COLAB_DRIVE_SAVE_PATH}")
except ImportError:
    print("[Kaggle GPU] Đang chạy trên môi trường Kaggle.")
    if DRIVE_FOLDER_ID:
        print(f"[Google Drive] Đang tải dữ liệu/trọng số hiện có từ Folder ID: {DRIVE_FOLDER_ID} ...")
        !gdown --folder https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID} -O training_lab/weights_drive_bundle --remaining-ok || true


In [ ]:
# BƯỚC 4: TỰ ĐỘNG SINH DỮ LIỆU SFT + DPO + HARD NEGATIVES TỪ KHO PDF
!python training_lab/kaggle_fn_v2/01_generate_sft_dpo_dataset.py


In [ ]:
# BƯỚC 5: CHẠY HUẤN LUYỆN TRỌN GÓI (TỰ ĐỘNG GIẢI PHÓNG VRAM GIỮA TỪNG AGENT & ÉP CHUẨN FP16 < 80MB)
# Ghi đè trực tiếp trọng số mới vào training_lab/weights_drive_bundle/
!python training_lab/kaggle_fn_v2/02_train_multilora_sft_dpo.py     --adapter {TRAIN_TARGET}     --dataset-dir training_lab/kaggle_fn_v2/datasets     --output-dir training_lab/weights_drive_bundle     --epochs {EPOCHS}


In [ ]:
# BƯỚC 6: TỰ ĐỘNG LƯU VỀ GOOGLE DRIVE LINK CỦA BẠN & TỰ ĐỘNG ĐẨY VỀ GITHUB (KHÔNG CẦN TẢI THỦ CÔNG)
import shutil

WEIGHTS_OUT = os.path.join(PROJECT_DIR, "training_lab", "weights_drive_bundle")
ZIP_BASE = os.path.join(PROJECT_DIR, "weights_bundle_v2")
zip_file = shutil.make_archive(ZIP_BASE, "zip", WEIGHTS_OUT)
print(f"✅ Đã đóng gói toàn bộ trọng số mới: {zip_file} ({os.path.getsize(zip_file)/1024/1024:.2f} MB)")

# 1. Tự động lưu vào Google Drive nếu chạy trên Colab (bao gồm cả Folder được share qua Link)
if IN_COLAB:
    dest_zip = os.path.join(COLAB_DRIVE_SAVE_PATH, "weights_bundle_v2.zip")
    shutil.copy2(zip_file, dest_zip)
    shutil.copytree(WEIGHTS_OUT, os.path.join(COLAB_DRIVE_SAVE_PATH, "weights_drive_bundle"), dirs_exist_ok=True)
    print(f"🎉 [TỰ ĐỘNG LƯU DRIVE THÀNH CÔNG] Đã lưu thẳng vào Google Drive: {COLAB_DRIVE_SAVE_PATH}")

    # Nếu bạn dán GOOGLE_DRIVE_FOLDER_LINK trên Colab, đẩy thẳng vào đúng Folder ID đó bằng Google Drive API v3:
    if DRIVE_FOLDER_ID:
        from google.colab import auth
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaFileUpload
        auth.authenticate_user()
        service = build("drive", "v3")
        meta = {"name": "weights_bundle_v2.zip", "parents": [DRIVE_FOLDER_ID]}
        media = MediaFileUpload(zip_file, mimetype="application/zip", resumable=True)
        uploaded = service.files().create(body=meta, media_body=media, fields="id, webViewLink").execute()
        print(f"🎉 [UPLOAD ĐÚNG LINK FOLDER DRIVE] File đã lên tại: {uploaded.get('webViewLink')}")

# 2. Tự động đẩy thẳng trọng số FP16 (<80MB/file) về lại GitHub Repo của bạn (Chạy được trên cả Kaggle & Colab!)
if GITHUB_TOKEN:
    print("[GitHub Auto-Push] Đang tự động commit và đẩy trọng số mới về GitHub...")
    auth_repo = f"https://{GITHUB_TOKEN}@github.com/ducmanh-jr/Multi-Agent_RAG.git"
    subprocess.run(["git", "config", "user.email", "trainer@kaggle-bot.local"], check=False)
    subprocess.run(["git", "config", "user.name", "Kaggle Auto Trainer"], check=False)
    subprocess.run(["git", "add", "training_lab/weights_drive_bundle"], check=False)
    subprocess.run(["git", "commit", "-m", "feat(weights): auto-update trained SFT+DPO+NLI weights from GPU session"], check=False)
    subprocess.run(["git", "push", auth_repo, "HEAD:main"], check=True)
    print("🎉 [TỰ ĐỘNG LƯU GITHUB THÀNH CÔNG] Trọng số mới đã nằm sẵn trên GitHub của bạn!")
